In [3]:
import urllib.request
import json

def test_sec_edgar(ticker: str):
    print(f"--- Fetching SEC EDGAR Data for {ticker.upper()} ---")
    
    headers = {
        'User-Agent': 'DataScience Portfolio Project (your.email@example.com)'
    }
    
    # SEC 8-K Item mappings for critical corporate events
    ITEM_MAPPING = {
        "1.01": "Entry into a Material Definitive Agreement",
        "1.02": "Termination of a Material Definitive Agreement",
        "1.03": "Bankruptcy or Receivership",
        "1.04": "Mine Safety - Reporting of Shutdowns and Patterns of Violations",
        "1.05": "Material Cybersecurity Incidents (Data Breach/Hack)",
        "2.01": "Completion of Acquisition or Disposition of Assets",
        "2.02": "Results of Operations and Financial Condition (Earnings Release)",
        "2.03": "Creation of a Direct Financial Obligation",
        "2.04": "Triggering Events That Accelerate or Increase a Direct Financial Obligation",
        "2.05": "Costs Associated with Exit or Disposal Activities (Layoffs/Restructuring)",
        "2.06": "Material Impairments",
        "3.01": "Notice of Delisting or Failure to Satisfy a Continued Listing Rule",
        "3.02": "Unregistered Sales of Equity Securities",
        "4.01": "Changes in Registrant's Certifying Accountant (Auditor Resigned/Dismissed)",
        "4.02": "Non-Reliance on Previously Issued Financial Statements (Accounting Error/Fraud)",
        "5.01": "Changes in Control of Registrant",
        "5.02": "Departure of Directors or Certain Officers (Executive Resigned/Fired)",
        "7.01": "Regulation FD Disclosure",
        "8.01": "Other Material Events",
        "9.01": "Financial Statements and Exhibits"
    }
    
    try:
        # Step 1: Map the Ticker to a CIK
        tickers_url = "https://www.sec.gov/files/company_tickers.json"
        req = urllib.request.Request(tickers_url, headers=headers)
        
        with urllib.request.urlopen(req) as response:
            tickers_data = json.loads(response.read().decode())
            
        cik_str = None
        for key, value in tickers_data.items():
            if value.get("ticker") == ticker.upper():
                cik_str = str(value.get("cik_str")).zfill(10)
                break
                
        if not cik_str:
            print(f"❌ Error: Could not find SEC CIK for ticker {ticker}")
            return
            
        # Step 2: Fetch the actual submissions data
        filings_url = f"https://data.sec.gov/submissions/CIK{cik_str}.json"
        req = urllib.request.Request(filings_url, headers=headers)
        
        with urllib.request.urlopen(req) as response:
            company_data = json.loads(response.read().decode())
            
        recent_filings = company_data.get("filings", {}).get("recent", {})
        
        # Step 3: Parse forms, dates, and the crucial 'items' array
        forms = recent_filings.get("form", [])
        dates = recent_filings.get("filingDate", [])
        items_list = recent_filings.get("items", [])
        
        print("\n--- Recent 8-K (Material Event) Filings ---")
        found = 0
        for i, form in enumerate(forms):
            if form == "8-K":
                # Get the comma-separated string of items (e.g., "2.02,9.01")
                raw_items = items_list[i] if i < len(items_list) else ""
                
                # Decode each item number into plain English
                decoded_events = []
                if raw_items:
                    for item_num in raw_items.split(","):
                        item_num = item_num.strip()
                        if item_num in ITEM_MAPPING:
                            decoded_events.append(ITEM_MAPPING[item_num])
                        else:
                            decoded_events.append(f"Item {item_num}")
                
                # Join them together for the LLM
                description = " | ".join(decoded_events) if decoded_events else "Unspecified Material Event"
                
                print(f"[{dates[i]}] {description}")
                found += 1
                
            if found >= 5: # Show top 5 for the test
                break
                
        if found == 0:
            print("No recent 8-K filings found.")
            
    except Exception as e:
        print(f"❌ Error: {str(e)}")

if __name__ == "__main__":
    test_sec_edgar("MSFT")

--- Fetching SEC EDGAR Data for MSFT ---

--- Recent 8-K (Material Event) Filings ---
[2026-06-05] Departure of Directors or Certain Officers (Executive Resigned/Fired)
[2026-05-14] Departure of Directors or Certain Officers (Executive Resigned/Fired) | Financial Statements and Exhibits
[2026-04-29] Results of Operations and Financial Condition (Earnings Release) | Financial Statements and Exhibits
[2026-01-28] Results of Operations and Financial Condition (Earnings Release) | Financial Statements and Exhibits
[2025-12-08] Departure of Directors or Certain Officers (Executive Resigned/Fired) | Item 5.07
